# Nifty 500: complete financial acquisition workflow

1. Download the official current Nifty 500 constituent list.
2. Attempt monthly NSE financial-filing metadata retrieval.
3. Filter to those index identities and standalone non-cumulative quarters;
   save a real XML download manifest.
4. Download referenced official XMLs with checkpoints.
5. Parse explicit financial concepts and XBRL contexts.
6. Export actual extracted values, quality reports and 16-quarter coverage.

Download/analysis period: **2022Q1–2025Q4 (16 calendar quarters)**.
Query filing metadata through today so later filings for 2025Q4 are included.
No OpenAI key is required. The NSE metadata endpoint is undocumented and may
return 403 in Colab. The notebook stops on failure and supports uploading source
metadata instead; it cannot guarantee unattended retrieval or complete coverage.
Current membership is not verified historical membership. The number of rows
in the official snapshot is retained as published, without truncating to 500.


In [1]:
from pathlib import Path

ROOT = Path("/content/nifty500_financial_research")
ROOT.mkdir(parents=True, exist_ok=True)
DOWNLOAD_START = "2022Q1"
DOWNLOAD_END = "2025Q4"
ANALYSIS_START = "2022Q1"
ANALYSIS_END = "2025Q4"
REQUIRED_FIELDS = ["PAT"]  # Add REVENUE, TOTAL_INCOME etc only when required.
MAX_DOWNLOADS = None  # Set 5 for a small trial; None processes all selected links.
PAUSE_SECONDS = 2


## 1. Load reusable download and analysis functions

The functions are embedded so this notebook runs without a repository or ZIP
installation. The same source is supplied separately as .py files.


In [10]:
import hashlib
import io
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit, unquote
from urllib.request import urlopen
from urllib.error import HTTPError
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd

INDEX_URL = "https://nsearchives.nseindia.com/content/indices/ind_nifty500list.csv"

def clean_text(series):
    return series.astype("string").str.strip().str.upper().replace("", pd.NA)

def read_index(content):
    reference = pd.read_csv(io.BytesIO(content), dtype="string", encoding="utf-8-sig")
    reference.columns = reference.columns.str.strip()
    reference = reference.rename(columns={
        "Company Name": "INDEX_COMPANY_NAME", "Symbol": "INDEX_SYMBOL",
        "ISIN Code": "INDEX_ISIN", "Industry": "INDEX_INDUSTRY",
    })
    required = {"INDEX_COMPANY_NAME", "INDEX_SYMBOL", "INDEX_ISIN", "INDEX_INDUSTRY"}
    if not required.issubset(reference):
        raise ValueError(f"Unexpected index CSV columns: {reference.columns.tolist()}")
    reference["INDEX_SYMBOL"] = clean_text(reference.INDEX_SYMBOL)
    reference["INDEX_ISIN"] = clean_text(reference.INDEX_ISIN)
    reference = reference.dropna(subset=["INDEX_SYMBOL", "INDEX_ISIN"]).copy()
    if reference.INDEX_SYMBOL.duplicated().any() or reference.INDEX_ISIN.duplicated().any():
        raise ValueError("Index identities are not unique; review the snapshot.")
    reference["COMPANY_KEY"] = "ISIN:" + reference.INDEX_ISIN
    reference["INDEX_SOURCE_URL"] = INDEX_URL
    reference["INDEX_RETRIEVED_AT_UTC"] = datetime.now(timezone.utc).isoformat()
    return reference


def company_name_key(values):
    return (values.astype("string").str.upper().str.strip()
            .str.replace(r"\bLTD\b", "LIMITED", regex=True)
            .str.replace(r"[^A-Z0-9]", "", regex=True))

def normalize_manifest(frame, reference):
    """Accept NSE website exports without pretending reference ISINs are source ISINs."""
    frame = frame.copy()
    frame.columns = frame.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()
    required = {"symbol", "isin", "consolidated", "cumulative", "period", "fromDate", "toDate", "xbrl"}
    if required.issubset(frame.columns):
        return frame
    aliases = {
        "COMPANY NAME": "companyName", "AUDITED / UNAUDITED": "audited",
        "CUMULATIVE / NON-CUMULATIVE": "cumulative",
        "CONSOLIDATED / NON-CONSOLIDATED": "consolidated",
        "IND AS/ NON IND AS": "indAs", "PERIOD": "period",
        "PERIOD ENDED": "toDate", "RELATING TO": "relatingTo",
        "XBRL": "xbrl", "EXCHANGE RECEIVED TIME": "filingDate",
        "EXCHANGE DISSEMINATION TIME": "broadCastDate",
        "SYMBOL": "symbol", "ISIN": "isin", "ISIN CODE": "isin",
        "PERIOD STARTED": "fromDate", "FROM DATE": "fromDate",
    }
    rename = {}
    for column in frame.columns:
        key = re.sub(r"\s+", " ", column.upper().replace("*", "")).strip()
        if key in aliases:
            rename[column] = aliases[key]
    frame = frame.rename(columns=rename)
    if not frame.columns.is_unique:
        raise ValueError("Export columns become ambiguous after normalization.")
    source_required = {"companyName", "consolidated", "cumulative", "period", "toDate", "xbrl"}
    if not source_required.issubset(frame):
        raise ValueError(f"Unsupported export: missing {sorted(source_required-set(frame.columns))}")
    if "symbol" not in frame:
        frame["symbol"] = pd.Series(pd.NA, index=frame.index, dtype="string")
    if "isin" not in frame:
        frame["isin"] = pd.Series(pd.NA, index=frame.index, dtype="string")
    frame["IDENTITY_ORIGIN"] = "SOURCE_IDENTIFIERS"
    ref = reference.copy()
    ref["_NAME_KEY"] = company_name_key(ref.INDEX_COMPANY_NAME)
    ref = ref.loc[~ref._NAME_KEY.duplicated(keep=False)]
    names = company_name_key(frame.companyName)
    missing_identity = clean_text(frame.symbol).isna() & clean_text(frame["isin"]).isna()
    inferred_symbol = names.map(ref.set_index("_NAME_KEY").INDEX_SYMBOL)
    matched = missing_identity & inferred_symbol.notna()
    frame.loc[matched, "symbol"] = inferred_symbol.loc[matched]
    frame.loc[matched, "IDENTITY_ORIGIN"] = "EXACT_NORMALIZED_COMPANY_NAME_REVIEW"
    # Do not populate missing source ISINs with current reference ISINs.
    # Quarter starts are metadata estimates and must be independently confirmed by XML.
    if "fromDate" not in frame:
        ends = pd.to_datetime(frame.toDate, format="mixed", errors="coerce")
        quarterly = clean_text(frame.period).eq("QUARTERLY")
        noncumulative = clean_text(frame.cumulative).isin(["NON-CUMULATIVE", "NON CUMULATIVE"])
        valid_end = ends.eq(ends.dt.to_period("Q").dt.end_time.dt.normalize())
        starts = ends.dt.to_period("Q").dt.start_time.where(quarterly & noncumulative & valid_end)
        frame["fromDate"] = starts.dt.strftime("%Y-%m-%d")
        frame["PERIOD_START_ORIGIN"] = "CALENDAR_QUARTER_START_REQUIRES_XML_CONFIRMATION"
    return frame

def prepare_filings(manifest, reference, start_quarter, end_quarter):
    manifest = normalize_manifest(manifest, reference)
    manifest.columns = manifest.columns.str.strip()
    required = {"symbol", "isin", "consolidated", "cumulative", "period",
                "fromDate", "toDate", "xbrl"}
    if not required.issubset(manifest):
        raise ValueError(f"Manifest missing: {sorted(required - set(manifest.columns))}")
    manifest["symbol"] = clean_text(manifest.symbol)
    manifest["isin"] = clean_text(manifest["isin"])
    manifest["_START"] = pd.to_datetime(manifest.fromDate, format="mixed", errors="coerce")
    manifest["_END"] = pd.to_datetime(manifest.toDate, format="mixed", errors="coerce")
    manifest["_QUARTER"] = manifest._END.dt.to_period("Q")
    expected = pd.period_range(start_quarter, end_quarter, freq="Q")
    basis = clean_text(manifest.consolidated)
    cumulative = clean_text(manifest.cumulative)
    duration = (manifest._END - manifest._START).dt.days + 1
    screen = (
        basis.isin(["STANDALONE", "NON-CONSOLIDATED", "NON CONSOLIDATED"])
        & cumulative.isin(["NON-CUMULATIVE", "NON CUMULATIVE"])
        & clean_text(manifest.period).eq("QUARTERLY")
        & manifest._QUARTER.isin(expected) & duration.between(70, 110)
    )
    screened = manifest.loc[screen].copy()
    by_isin = reference.set_index("INDEX_ISIN").COMPANY_KEY
    by_symbol = reference.set_index("INDEX_SYMBOL").COMPANY_KEY
    screened["COMPANY_KEY"] = screened["isin"].map(by_isin)
    screened["IDENTITY_MATCH"] = pd.Series(pd.NA, index=screened.index, dtype="string")
    screened.loc[screened.COMPANY_KEY.notna(), "IDENTITY_MATCH"] = "EXACT_ISIN"
    # Symbol-only matching is allowed only when the manifest lacks a valid ISIN.
    has_isin = screened["isin"].str.fullmatch(r"IN[A-Z0-9]{10}", na=False)
    fallback = ~has_isin & screened.COMPANY_KEY.isna()
    screened.loc[fallback, "COMPANY_KEY"] = screened.loc[fallback, "symbol"].map(by_symbol)
    screened.loc[fallback & screened.COMPANY_KEY.notna(), "IDENTITY_MATCH"] = "SYMBOL_ONLY_REVIEW"
    if "IDENTITY_ORIGIN" in screened:
        name_match = screened.IDENTITY_ORIGIN.eq("EXACT_NORMALIZED_COMPANY_NAME_REVIEW") & screened.COMPANY_KEY.notna()
        screened.loc[name_match, "IDENTITY_MATCH"] = "EXACT_COMPANY_NAME_REVIEW"

    symbol_conflict = (
        has_isin & screened.COMPANY_KEY.isna()
        & screened.symbol.isin(reference.INDEX_SYMBOL)
    )
    conflicts = screened.loc[symbol_conflict].copy()
    selected = screened.loc[screened.COMPANY_KEY.notna()].copy()
    selected = selected.merge(reference, on="COMPANY_KEY", how="left", validate="many_to_one")
    selected["xbrl"] = selected.xbrl.astype("string").str.strip()
    # Exact repeated URLs for the same company/period are one download job.
    selected = selected.drop_duplicates(["COMPANY_KEY", "_START", "_END", "xbrl"])
    return selected, conflicts

def validate_xml(content):
    if not content or re.search(br"<!\s*(?:DOCTYPE|ENTITY)\b", content, re.I):
        raise ValueError("Empty file or unsupported DTD/entity declaration")
    root = ET.fromstring(content)
    if root.tag != "{http://www.xbrl.org/2003/instance}xbrl":
        raise ValueError("Not an XBRL instance XML; HTML/PDF/Inline XBRL requires another reader")
    return root

def allowed_nse_url(url):
    try:
        parsed = urlsplit(str(url))
        return (parsed.scheme == "https" and not parsed.username and not parsed.password
                and parsed.hostname in {"nsearchives.nseindia.com", "archives.nseindia.com"}
                and Path(parsed.path).suffix.lower() in {".xml", ".xbrl"})
    except ValueError:
        return False

# def download_filings(selected, xml_dir, log_path, max_downloads=None, pause_seconds=2):
#     xml_dir, log_path = Path(xml_dir), Path(log_path)
#     xml_dir.mkdir(parents=True, exist_ok=True)
#     jobs = selected.dropna(subset=["xbrl"]).drop_duplicates("xbrl")
#     if max_downloads is not None:
#         jobs = jobs.head(max_downloads)
#     logs = []
#     if True:
#         for count, url in enumerate(jobs.xbrl, start=1):
#             entry = {"SOURCE_XBRL_URL": url, "DOWNLOAD_STATUS": "NOT_ATTEMPTED",
#                      "LOCAL_XML": "", "SHA256": "", "DETAIL": ""}
#             should_stop = False
#             try:
#                 if not allowed_nse_url(url):
#                     raise ValueError("Unsupported URL; use an official NSE archive XML link")
#                 name = re.sub(r"[^A-Za-z0-9_.-]", "_", unquote(Path(urlsplit(url).path).stem))
#                 digest = hashlib.sha256(url.encode()).hexdigest()[:16]
#                 local = xml_dir / f"{name}_{digest}.xml"
#                 if local.exists():
#                     content = local.read_bytes()
#                     validate_xml(content)
#                     entry["DOWNLOAD_STATUS"] = "CACHED_VALID_XML"
#                 else:
#                     with urlopen(url, timeout=60) as response:
#                         if not allowed_nse_url(response.geturl()):
#                             raise ValueError("Redirected outside the supported archive URLs")
#                         content = response.read()
#                         validate_xml(content)
#                         local.write_bytes(content)
#                         entry["DOWNLOAD_STATUS"] = "DOWNLOADED"
#                 if entry["DOWNLOAD_STATUS"] in {"CACHED_VALID_XML", "DOWNLOADED"}:
#                     entry["LOCAL_XML"] = str(local)
#                     entry["SHA256"] = hashlib.sha256(content).hexdigest()
#             except HTTPError as exc:
#                 if exc.code in {401, 403, 429}:
#                     entry["DOWNLOAD_STATUS"] = f"ACCESS_STOPPED_{exc.code}"
#                     should_stop = True
#                 else:
#                     entry["DOWNLOAD_STATUS"] = "FAILED"
#                     entry["DETAIL"] = f"HTTP {exc.code}"
#             except Exception as exc:
#                 entry["DOWNLOAD_STATUS"] = "FAILED"
#                 entry["DETAIL"] = f"{type(exc).__name__}: {str(exc)[:250]}"
#             entry["CHECKED_AT_UTC"] = datetime.now(timezone.utc).isoformat()
#             logs.append(entry)
#             pd.DataFrame(logs).to_csv(log_path, index=False)
#             if should_stop:
#                 print("Stopped on access restriction; completed downloads are retained.")
#                 break
#             if count % 25 == 0:
#                 print(f"Processed {count} / {len(jobs)} unique filing URLs")
#             time.sleep(pause_seconds)
#     return pd.DataFrame(logs, columns=["SOURCE_XBRL_URL", "DOWNLOAD_STATUS", "LOCAL_XML",
#                                       "SHA256", "DETAIL", "CHECKED_AT_UTC"])

def download_filings(
    selected,
    xml_dir,
    log_path,
    max_downloads=None,
    pause_seconds=2,
    timeout_seconds=15,
    max_consecutive_failures=3,
):
    from urllib.request import Request, urlopen
    from urllib.error import HTTPError

    xml_dir = Path(xml_dir)
    log_path = Path(log_path)
    xml_dir.mkdir(parents=True, exist_ok=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    jobs = selected.dropna(subset=["xbrl"]).copy()
    jobs["xbrl"] = jobs["xbrl"].astype("string").str.strip()
    jobs = jobs.loc[jobs["xbrl"].ne("")].drop_duplicates("xbrl")

    if max_downloads is not None:
        jobs = jobs.head(max_downloads)

    columns = [
        "SOURCE_XBRL_URL", "DOWNLOAD_STATUS", "LOCAL_XML",
        "SHA256", "DETAIL", "CHECKED_AT_UTC",
    ]
    logs = []
    successful = 0
    failed = 0
    consecutive_failures = 0
    success_statuses = {"DOWNLOADED", "CACHED_VALID_XML"}

    print(f"Unique XML URLs to process: {len(jobs):,}", flush=True)

    for count, url in enumerate(jobs["xbrl"], start=1):
        print(f"\n[{count}/{len(jobs)}] Starting: {url}", flush=True)

        entry = {
            "SOURCE_XBRL_URL": url,
            "DOWNLOAD_STATUS": "NOT_ATTEMPTED",
            "LOCAL_XML": "",
            "SHA256": "",
            "DETAIL": "",
        }
        should_stop = False

        try:
            if not allowed_nse_url(url):
                raise ValueError("Unsupported official NSE XML URL")

            name = re.sub(
                r"[^A-Za-z0-9_.-]", "_",
                unquote(Path(urlsplit(url).path).stem),
            )
            digest = hashlib.sha256(url.encode()).hexdigest()[:16]
            local = xml_dir / f"{name}_{digest}.xml"

            if local.exists():
                content = local.read_bytes()
                validate_xml(content)
                entry["DOWNLOAD_STATUS"] = "CACHED_VALID_XML"
            else:
                request = Request(url, headers={
                    "User-Agent": "Mozilla/5.0",
                    "Accept": "application/xml,text/xml,*/*",
                    "Referer": "https://www.nseindia.com/",
                })

                with urlopen(request, timeout=timeout_seconds) as response:
                    if not allowed_nse_url(response.geturl()):
                        raise ValueError("Unsupported redirect destination")
                    content = response.read()

                validate_xml(content)
                local.write_bytes(content)
                entry["DOWNLOAD_STATUS"] = "DOWNLOADED"

            entry["LOCAL_XML"] = str(local)
            entry["SHA256"] = hashlib.sha256(content).hexdigest()

        except HTTPError as exc:
            entry["DETAIL"] = f"HTTP {exc.code}: {exc.reason}"
            if exc.code in {401, 403, 429}:
                entry["DOWNLOAD_STATUS"] = f"ACCESS_STOPPED_{exc.code}"
                should_stop = True
            else:
                entry["DOWNLOAD_STATUS"] = "FAILED"

        except Exception as exc:
            entry["DOWNLOAD_STATUS"] = "FAILED"
            entry["DETAIL"] = f"{type(exc).__name__}: {str(exc)[:250]}"

        entry["CHECKED_AT_UTC"] = datetime.now(timezone.utc).isoformat()
        logs.append(entry)
        pd.DataFrame(logs, columns=columns).to_csv(log_path, index=False)

        if entry["DOWNLOAD_STATUS"] in success_statuses:
            successful += 1
            consecutive_failures = 0
        else:
            failed += 1
            consecutive_failures += 1

        print(
            f"Result: {entry['DOWNLOAD_STATUS']}"
            + (f" | {entry['DETAIL']}" if entry["DETAIL"] else ""),
            flush=True,
        )
        print(
            f"Processed: {count}/{len(jobs)} | "
            f"Successful: {successful} | Failed: {failed}",
            flush=True,
        )

        if should_stop:
            print("Stopped on an access restriction.", flush=True)
            break

        if (
            max_consecutive_failures is not None
            and consecutive_failures >= max_consecutive_failures
        ):
            print(
                f"Stopped after {consecutive_failures} consecutive failures. "
                "Check connectivity before continuing.",
                flush=True,
            )
            break

        if count < len(jobs):
            time.sleep(pause_seconds)

    print(
        f"\nFinished: {successful} successful, {failed} failed, "
        f"{len(jobs) - len(logs)} unattempted.",
        flush=True,
    )
    print(f"Log saved: {log_path}", flush=True)

    return pd.DataFrame(logs, columns=columns)


def extract_downloads(log, selected, parse_function, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    extracted, issues, summaries, errors = [], [], [], []
    metadata = {url: group for url, group in selected.groupby("xbrl")}
    valid = log.loc[log.DOWNLOAD_STATUS.isin(["DOWNLOADED", "CACHED_VALID_XML"])]
    for item in valid.to_dict("records"):
        path, url = Path(item["LOCAL_XML"]), item["SOURCE_XBRL_URL"]
        group = metadata[url]
        if len(group[["COMPANY_KEY", "_START", "_END"]].drop_duplicates()) != 1:
            errors.append({"SOURCE_XBRL_URL": url, "ERROR": "URL has conflicting company/period metadata"})
            continue
        meta = group.iloc[0]
        try:
            # Original filename preserves the filing-format prefix expected by the parser.
            original_name = unquote(Path(urlsplit(url).path).name)
            rows, facts, file_issues, summary = parse_function(original_name, path.read_bytes())
            for row in rows:
                row.update({
                    "COMPANY_KEY": meta.COMPANY_KEY, "INDEX_SYMBOL": meta.INDEX_SYMBOL,
                    "INDEX_COMPANY_NAME": meta.INDEX_COMPANY_NAME,
                    "INDEX_INDUSTRY": meta.INDEX_INDUSTRY, "INDEX_ISIN": meta.INDEX_ISIN,
                    "IDENTITY_MATCH": meta.IDENTITY_MATCH,
                    "MANIFEST_SYMBOL": meta.symbol, "MANIFEST_ISIN": meta["isin"],
                    "MANIFEST_START": str(meta._START.date()),
                    "MANIFEST_END": str(meta._END.date()),
                    "SOURCE_XBRL_URL": url, "SOURCE_SHA256": item["SHA256"],
                })
                row["XML_SYMBOL_CONFIRMED"] = (
                    bool(row.get("NSE_SYMBOL"))
                    and str(row["NSE_SYMBOL"]).strip().upper() == str(meta.INDEX_SYMBOL).strip().upper()
                )
                # No financial value, date, basis or quarter flag is filled from metadata.
                extracted.append(row)
            issues.extend({**r, "SOURCE_XBRL_URL": url} for r in file_issues)
            summaries.append({**summary, "SOURCE_XBRL_URL": url})
        except Exception as exc:
            errors.append({"SOURCE_XBRL_URL": url, "ERROR": f"{type(exc).__name__}: {str(exc)[:250]}"})
    context_rows = pd.DataFrame(extracted)
    context_rows.to_csv(destination / "all_extracted_contexts.csv", index=False)
    pd.DataFrame(issues).to_csv(destination / "extraction_issues.csv", index=False)
    pd.DataFrame(summaries).to_csv(destination / "filing_summary.csv", index=False)
    pd.DataFrame(errors).to_csv(destination / "extraction_errors.csv", index=False)
    return context_rows

def build_panel(context_rows, reference, start_quarter, end_quarter, required_fields):
    expected = pd.period_range(start_quarter, end_quarter, freq="Q")
    candidates = context_rows.copy()
    if candidates.empty:
        candidates = pd.DataFrame(columns=["COMPANY_KEY", "PERIOD_START", "PERIOD_END",
            "MANIFEST_START", "MANIFEST_END", "STANDALONE_CONSOLIDATED",
            "STRICT_QUARTER_CANDIDATE", "CONTEXT_DEFINED", "IDENTITY_MATCH", *required_fields])
    candidates["_date"] = pd.to_datetime(candidates.PERIOD_END, errors="coerce")
    candidates["_start"] = pd.to_datetime(candidates.PERIOD_START, errors="coerce")
    candidates["_quarter"] = candidates._date.dt.to_period("Q")
    context_flag = clean_text(candidates.CONTEXT_DEFINED).isin(["TRUE", "1"])
    strict_flag = clean_text(candidates.STRICT_QUARTER_CANDIDATE).isin(["TRUE", "1"])
    same_period = (candidates._date.eq(pd.to_datetime(candidates.MANIFEST_END, errors="coerce"))
                   & candidates._start.eq(pd.to_datetime(candidates.MANIFEST_START, errors="coerce")))
    identity_ok = candidates.IDENTITY_MATCH.eq("EXACT_ISIN")
    if "XML_SYMBOL_CONFIRMED" in candidates:
        identity_ok = identity_ok | (
            candidates.IDENTITY_MATCH.isin(["EXACT_COMPANY_NAME_REVIEW", "SYMBOL_ONLY_REVIEW"])
            & candidates.XML_SYMBOL_CONFIRMED.fillna(False).eq(True))
    mask = (context_flag & strict_flag & same_period
            & clean_text(candidates.STANDALONE_CONSOLIDATED).eq("STANDALONE")
            & candidates._quarter.isin(expected) & identity_ok)
    candidates = candidates.loc[mask].copy()
    for field in required_fields:
        if field not in candidates:
            candidates[field] = np.nan
        candidates[field] = pd.to_numeric(candidates[field], errors="coerce").replace([np.inf, -np.inf], np.nan)
    duplicates = candidates.duplicated(["COMPANY_KEY", "_quarter"], keep=False)
    review = candidates.loc[duplicates].copy()
    usable = candidates.loc[~duplicates].copy()
    usable["_complete"] = usable[required_fields].notna().all(axis=1)
    if usable.empty:
        matrix = pd.DataFrame(False, index=pd.Index(reference.COMPANY_KEY, name="COMPANY_KEY"), columns=expected)
    else:
        matrix = usable.pivot(index="COMPANY_KEY", columns="_quarter", values="_complete")
        matrix = matrix.reindex(index=reference.COMPANY_KEY, columns=expected).fillna(False).astype(bool)
    coverage = reference.set_index("COMPANY_KEY")[["INDEX_SYMBOL", "INDEX_COMPANY_NAME", "INDEX_INDUSTRY"]].copy()
    coverage["complete_quarters"] = matrix.sum(axis=1)
    coverage["required_quarters"] = len(expected)
    coverage["eligible"] = matrix.all(axis=1)
    coverage["missing_or_invalid_quarters"] = matrix.apply(
        lambda row: ", ".join(str(q) for q in row.index[~row]), axis=1)
    eligible = coverage.index[coverage.eligible]
    complete = usable.loc[usable.COMPANY_KEY.isin(eligible)].sort_values(["COMPANY_KEY", "_date"])
    return usable, complete, coverage, review


from urllib.parse import urlencode

class MetadataUnavailable(RuntimeError):
    pass

def retrieve_filing_metadata(folder, start_date, end_date, pause_seconds=2):
    """Attempt NSE website metadata; stop on access or schema failures.

    The website endpoint is undocumented and is not a guaranteed public API.
    Query windows are filing/publication dates, not financial quarter dates.
    """
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    first, last = pd.Timestamp(start_date), pd.Timestamp(end_date)
    if last < first:
        raise ValueError("Metadata end date precedes start date.")
    raw, query_log = [], []
    windows = pd.period_range(first, last, freq="M")
    required = {"symbol", "isin", "consolidated", "cumulative", "period", "fromDate", "toDate", "xbrl"}
    for month in windows:
        begin, finish = max(first, month.start_time), min(last, month.end_time.normalize())
        params = {
            "index": "equities", "from_date": begin.strftime("%d-%m-%Y"),
            "to_date": finish.strftime("%d-%m-%Y"),
        }
        url = "https://www.nseindia.com/api/corporates-financial-results?" + urlencode(params)
        cache = folder / f"metadata_{month}.json"
        try:
            if cache.exists():
                payload = json.loads(cache.read_text(encoding="utf-8"))
                status = "CACHED"
            else:
                payload = request_metadata_json(url)
                status = "DOWNLOADED"
            records = payload if isinstance(payload, list) else payload.get("data") if isinstance(payload, dict) else None
            if not isinstance(records, list):
                raise ValueError("Unexpected metadata response structure; review endpoint schema")
            if records and any(not isinstance(r, dict) or not required.issubset(r) for r in records):
                raise ValueError("Metadata is missing required source fields")
            if not cache.exists():
                cache.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
            raw.extend(records)
            query_log.append({"month":str(month), "url":url, "status":status, "records":len(records)})
            pd.DataFrame(query_log).to_csv(folder / "metadata_query_log.csv", index=False)
            pd.DataFrame(raw).to_csv(folder / "filing_metadata_partial.csv", index=False)
            print(f"Metadata {month}: {len(records)} records")
        except Exception as exc:
            query_log.append({"month":str(month), "url":url,
                              "status":f"STOPPED_{getattr(exc, 'code', type(exc).__name__)}", "records":0})
            pd.DataFrame(query_log).to_csv(folder / "metadata_query_log.csv", index=False)
            pd.DataFrame(raw).to_csv(folder / "filing_metadata_partial.csv", index=False)
            raise MetadataUnavailable(
                "Metadata retrieval stopped. Completed records were saved. "
                "Use METADATA_SOURCE='upload' with source filing metadata to continue. "
                "Do not interpret partial queries as complete quarter coverage."
            ) from exc
        time.sleep(pause_seconds)
    result = pd.DataFrame(raw)
    if result.empty:
        raise MetadataUnavailable("No filing metadata returned. Upload a manifest instead.")
    result.to_csv(folder / "filing_metadata_all.csv", index=False)
    return result


def request_metadata_json(url, attempts=3):
    """Retry transient transport/server failures; stop on access restrictions."""
    import socket
    from urllib.error import URLError, HTTPError
    for attempt in range(attempts):
        try:
            with urlopen(url, timeout=120) as response:
                return json.loads(response.read().decode("utf-8-sig"))
        except HTTPError as exc:
            if exc.code not in {408, 500, 502, 503, 504} or attempt == attempts - 1:
                raise
        except (TimeoutError, socket.timeout, URLError):
            if attempt == attempts - 1:
                raise
        print(f"Temporary metadata failure; retry {attempt + 1}/{attempts - 1}")
        time.sleep(5 * (attempt + 1))


original_normalize_manifest = normalize_manifest

def normalize_manifest(frame, reference):
    # Canonical names are case-sensitive: do not lowercase the whole dataframe.
    frame = frame.copy()
    # Match export headers regardless of case, whitespace, slashes or underscores.
    export_aliases = {
        "COMPANYNAME": "companyName", "SYMBOL": "symbol",
        "ISIN": "isin", "ISINCODE": "isin",
        "AUDITEDUNAUDITED": "audited",
        "CONSOLIDATEDNONCONSOLIDATED": "consolidated",
        "CUMULATIVENONCUMULATIVE": "cumulative",
        "PERIOD": "period", "PERIODENDED": "toDate",
        "PERIODSTARTED": "fromDate", "FROMDATE": "fromDate",
        "TODATE": "toDate", "XBRL": "xbrl", "RELATINGTO": "relatingTo",
    }
    frame.columns = [c if c.startswith("_") else export_aliases.get(re.sub(r"[^A-Z0-9]", "", c.upper()), c)
                     for c in frame.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()]
    canonical = ["symbol", "isin", "companyName", "consolidated", "cumulative",
                 "period", "fromDate", "toDate", "xbrl", "filingDate", "broadCastDate"]
    lookup = {name.lower(): name for name in canonical}
    frame.columns = [lookup.get(str(c).replace("\ufeff", "").strip().lower(),
                                str(c).replace("\ufeff", "").strip()) for c in frame.columns]
    if not frame.columns.is_unique:
        raise ValueError("Duplicate column names after case normalization")
    keys = {re.sub(r"[^A-Z0-9]", "", c.upper()): c for c in frame.columns}
    # The Integrated Filing CSV supplies quarter-end and basis, but often no
    # duration/cumulative flag. Keep those entries as discovery candidates only.
    integrated = "QUARTERENDDATE" in keys or "FORQUARTERENDING" in keys
    if integrated:
        aliases = {"QUARTERENDDATE": "toDate", "FORQUARTERENDING": "toDate",
                   "CONSOLIDATEDSTANDALONE": "consolidated", "COMPANYNAME": "companyName",
                   "SYMBOL": "symbol", "ISIN": "isin", "XBRL": "xbrl",
                   "BROADCASTDATETIME": "broadCastDate"}
        frame = frame.rename(columns={col: aliases[key] for key, col in keys.items() if key in aliases})
        if not frame.columns.is_unique:
            raise ValueError("Ambiguous Integrated Filing columns")
        for required in ["toDate", "consolidated", "companyName", "xbrl"]:
            if required not in frame:
                raise ValueError(f"Integrated Filing export missing {required}; provide the original CSV with XML URLs")
        frame["METADATA_SOURCE_KIND"] = "INTEGRATED_FILING"
        frame["METADATA_PERIOD_STATUS"] = "QUARTER_END_ONLY_REQUIRES_XML_CONFIRMATION"
        ends = pd.to_datetime(frame.toDate, format="mixed", errors="coerce")
        valid_end = ends.eq(ends.dt.to_period("Q").dt.end_time.dt.normalize())
        frame["fromDate"] = ends.dt.to_period("Q").dt.start_time.where(valid_end).dt.strftime("%Y-%m-%d")
        # These are candidate-selection estimates, never source assertions.
        frame["period"] = "Quarterly"
        frame["cumulative"] = "Non-cumulative"
        frame["PERIOD_START_ORIGIN"] = "CALENDAR_QUARTER_START_REQUIRES_XML_CONFIRMATION"
    result = original_normalize_manifest(frame, reference)
    if integrated:
        result["METADATA_PERIOD_STATUS"] = "QUARTER_END_ONLY_REQUIRES_XML_CONFIRMATION"
    return result


def upload_source_metadata(reference):
    from google.colab import files
    print("Upload original Financial Results CSVs AND Integrated Filing - Financials CSVs.")
    print("Exports must contain actual XBRL XML URLs, not only a View label.")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No source metadata uploaded; no manifest was generated")
    frames = []
    for name, content in uploaded.items():
        frame = pd.read_csv(io.BytesIO(content), dtype="string", encoding="utf-8-sig")
        frame = normalize_manifest(frame, reference)
        frame["MANIFEST_INPUT_NAME"] = name
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


## 2. Download the current official Nifty 500 list

If the direct download is blocked, set USE_UPLOADED_INDEX=True and upload the
CSV downloaded manually from the official Nifty 500 page. Keep the snapshot.
Do not represent its labels or membership as verified historical classifications.


In [3]:
USE_UPLOADED_INDEX = True

if USE_UPLOADED_INDEX:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one Nifty 500 constituent CSV.")
    index_bytes = next(iter(uploaded.values()))
else:
    with urlopen(INDEX_URL, timeout=60) as result:
        index_bytes = result.read()

nifty500 = read_index(index_bytes)
(ROOT / "nifty500_original_download.csv").write_bytes(index_bytes)
nifty500.to_csv(ROOT / "nifty500_constituent_snapshot.csv", index=False)
print("Companies in current snapshot:", len(nifty500))
display(nifty500.head())


Saving ind_nifty500list.csv to ind_nifty500list.csv
Companies in current snapshot: 501


,INDEX_COMPANY_NAME,INDEX_INDUSTRY,INDEX_SYMBOL,Series,INDEX_ISIN,COMPANY_KEY,INDEX_SOURCE_URL,INDEX_RETRIEVED_AT_UTC
0,360 ONE WAM Ltd.,Financial Services,360ONE,EQ,INE466L01038,ISIN:INE466L01038,https://nsearchives.nseindia.com/content/indic...,2026-10-18T17:44:15.024358+00:00
1,ABB India Ltd.,Capital Goods,ABB,EQ,INE117A01022,ISIN:INE117A01022,https://nsearchives.nseindia.com/content/indic...,2026-10-18T17:44:15.024358+00:00
2,ACC Ltd.,Construction Materials,ACC,EQ,INE012A01025,ISIN:INE012A01025,https://nsearchives.nseindia.com/content/indic...,2026-10-18T17:44:15.024358+00:00
3,ACME Solar Holdings Ltd.,Power,ACMESOLAR,EQ,INE622W01025,ISIN:INE622W01025,https://nsearchives.nseindia.com/content/indic...,2026-10-18T17:44:15.024358+00:00
4,AIA Engineering Ltd.,Capital Goods,AIAENG,EQ,INE212H01026,ISIN:INE212H01026,https://nsearchives.nseindia.com/content/indic...,2026-10-18T17:44:15.024358+00:00


## 3. Generate the 2022Q1–2025Q4 download manifest

Default `auto` retries transient NSE failures (three attempts, 120-second timeout),
reuses cached completed months, and offers CSV upload if discovery fails.
Access restrictions (401/403/429) stop automatic retrieval immediately.

Upload original CSV exports from both official sources:
- https://www.nseindia.com/companies-listing/corporate-filings-financial-results
- https://www.nseindia.com/companies-listing/corporate-integrated-filing

NSE directs March 2025 onward results to Integrated Filing. Automatic retrieval
here queries the legacy endpoint only; the second source is supplied by CSV.
Keep ADD_SOURCE_EXPORTS=True for the requested period. Include publication dates
in 2026 for December 2025 results. Export in smaller date windows if necessary.
No pre-existing download manifest is required. Use METADATA_SOURCE="upload" to
skip the automatic request when your runtime cannot reach NSE.

Integrated Filing rows that specify only a quarter end use an estimated quarter
start and candidate screening flags, explicitly marked in METADATA_PERIOD_STATUS.
These flags do not prove non-cumulative quarterly values. XML context validation
remains mandatory. Unsupported URLs are excluded and reported. A missing-pairs
CSV reports every company/quarter lacking a candidate link; it does not assess
successful extraction. Current index membership is not historical membership.


In [6]:
METADATA_SOURCE = "upload"  # auto: retry NSE, then offer upload; upload: skip NSE
ADD_SOURCE_EXPORTS = False  # Include Integrated Filing exports for 2025.

manifest = None
if METADATA_SOURCE in {"auto", "nse"}:
    try:
        manifest = retrieve_filing_metadata(
            ROOT / "metadata", start_date="2022-01-01",
            end_date=pd.Timestamp.now(tz="Asia/Kolkata").date().isoformat(),
            pause_seconds=PAUSE_SECONDS,
        )
    except MetadataUnavailable as exc:
        print(str(exc))
        if METADATA_SOURCE == "nse":
            raise
        print("Automatic discovery failed. Upload source exports to continue.")
elif METADATA_SOURCE != "upload":
    raise ValueError("METADATA_SOURCE must be auto, nse, or upload")

if manifest is None or ADD_SOURCE_EXPORTS:
    supplied = upload_source_metadata(nifty500)
    manifest = supplied if manifest is None else pd.concat(
        [normalize_manifest(manifest, nifty500), supplied], ignore_index=True)

manifest = normalize_manifest(manifest, nifty500)
manifest.to_csv(ROOT / "manifest_input_combined.csv", index=False)

selected_filings, identity_conflicts = prepare_filings(
    manifest, nifty500, DOWNLOAD_START, DOWNLOAD_END)
selected_filings.to_csv(ROOT / "Nifty500_Download_Manifest_2022_2025.csv", index=False)
identity_conflicts.to_csv(ROOT / "identity_conflicts_review.csv", index=False)

# Drop unusable links and report them separately before downstream downloads.
valid_links = selected_filings.xbrl.map(allowed_nse_url)
selected_filings.loc[~valid_links].to_csv(ROOT / "invalid_manifest_urls.csv", index=False)
selected_filings = selected_filings.loc[valid_links].copy()
selected_filings.to_csv(ROOT / "Nifty500_Download_Manifest_2022_2025.csv", index=False)
found = set(selected_filings.COMPANY_KEY)
missing_from_manifest = nifty500.loc[~nifty500.COMPANY_KEY.isin(found)]
missing_from_manifest.to_csv(ROOT / "companies_missing_from_manifest.csv", index=False)

print("Matching companies with filing metadata:", len(found))
print("Unique XML URLs:", selected_filings.xbrl.nunique())
print("Companies without matching metadata:", len(missing_from_manifest))
print("Identity conflicts requiring review:", len(identity_conflicts))
expected_download_quarters = pd.period_range(DOWNLOAD_START, DOWNLOAD_END, freq="Q")
filing_quarter_counts = (
    selected_filings.groupby("_QUARTER").COMPANY_KEY.nunique()
    .reindex(expected_download_quarters, fill_value=0)
    .rename("companies_with_filing_metadata")
)
print("Filing metadata coverage by calendar quarter:")
display(filing_quarter_counts.to_frame())
filing_quarter_counts.to_csv(ROOT / "manifest_quarter_coverage.csv")
missing_metadata_quarters = filing_quarter_counts.index[filing_quarter_counts.eq(0)]
if len(missing_metadata_quarters):
    print("WARNING: No matching metadata for:", ", ".join(map(str, missing_metadata_quarters)))
    print("Supply those filing URLs in an additional manifest; they cannot be downloaded otherwise.")
display(missing_from_manifest[["INDEX_SYMBOL", "INDEX_COMPANY_NAME"]])

print("Manifest saved:", ROOT / "Nifty500_Download_Manifest_2022_2025.csv")

# A populated quarter does not establish coverage for all index companies.
grid = pd.MultiIndex.from_product(
    [nifty500.COMPANY_KEY, expected_download_quarters], names=["COMPANY_KEY", "_QUARTER"])
present = pd.MultiIndex.from_frame(selected_filings[["COMPANY_KEY", "_QUARTER"]].drop_duplicates())
missing_pairs = grid.difference(present).to_frame(index=False)
missing_pairs = missing_pairs.merge(nifty500[["COMPANY_KEY", "INDEX_SYMBOL", "INDEX_COMPANY_NAME"]], on="COMPANY_KEY")
missing_pairs.to_csv(ROOT / "missing_company_quarters.csv", index=False)
print("Company-quarter pairs without candidate XML links:", len(missing_pairs))
print("Manifest rows are discovery candidates; financial XML must confirm standalone quarterly values.")
print("All downstream extraction and strict context checks remain required.")


Upload original Financial Results CSVs AND Integrated Filing - Financials CSVs.
Exports must contain actual XBRL XML URLs, not only a View label.


Saving CF-FR-equities-01-01-2022-to-31-12-2022.csv to CF-FR-equities-01-01-2022-to-31-12-2022.csv
Saving CF-FR-equities-01-01-2026-to-18-10-2026.csv to CF-FR-equities-01-01-2026-to-18-10-2026 (1).csv
Saving CF-FR-equities-01-01-2023-to-31-12-2023.csv to CF-FR-equities-01-01-2023-to-31-12-2023 (1).csv
Saving CF-FR-equities-01-01-2025-to-31-12-2025.csv to CF-FR-equities-01-01-2025-to-31-12-2025 (1).csv
Saving CF-FR-equities-01-01-2024-to-31-12-2024.csv to CF-FR-equities-01-01-2024-to-31-12-2024 (1).csv
Matching companies with filing metadata: 435
Unique XML URLs: 4744
Companies without matching metadata: 66
Identity conflicts requiring review: 0
Filing metadata coverage by calendar quarter:


,companies_with_filing_metadata
2022Q1,379
2022Q2,278
2022Q3,366
2022Q4,383
2023Q1,391
2023Q2,396
2023Q3,404
2023Q4,409
2024Q1,415
2024Q2,420


Supply those filing URLs in an additional manifest; they cannot be downloaded otherwise.


,INDEX_SYMBOL,INDEX_COMPANY_NAME
11,ABBOTINDIA,Abbott India Ltd.
23,CPPLUS,Aditya Infotech Ltd.
25,AEGISVOPAK,Aegis Vopak Terminals Ltd.
29,AJANTPHARM,Ajanta Pharmaceuticals Ltd.
38,ANTHEM,Anthem Biosciences Ltd.
...,...,...
478,VAML,Vedanta Aluminium Metal Ltd.
479,VISL,Vedanta Iron and Steel Ltd.
481,VOGL,Vedanta Oil and Gas Ltd.
482,VEDPOWER,Vedanta Power Ltd.


Manifest saved: /content/nifty500_financial_research/Nifty500_Download_Manifest_2022_2025.csv
Company-quarter pairs without candidate XML links: 3313
Manifest rows are discovery candidates; financial XML must confirm standalone quarterly values.
All downstream extraction and strict context checks remain required.


In [9]:
selected_filings.head()



,companyName,audited,cumulative,consolidated,indAs,period,toDate,relatingTo,xbrl,filingDate,...,_QUARTER,COMPANY_KEY,IDENTITY_MATCH,INDEX_COMPANY_NAME,INDEX_INDUSTRY,INDEX_SYMBOL,Series,INDEX_ISIN,INDEX_SOURCE_URL,INDEX_RETRIEVED_AT_UTC
0,Inox Wind Limited,Un-Audited,Non-cumulative,Non-Consolidated,Ind-AS New,Quarterly,30-Sep-2022,Second Quarter,https://nsearchives.nseindia.com/corporate/xbr...,03-Dec-2022 15:27:49,...,2022Q3,ISIN:INE066P01011,EXACT_COMPANY_NAME_REVIEW,Inox Wind Ltd.,Capital Goods,INOXWIND,EQ,INE066P01011,https://nsearchives.nseindia.com/content/indic...,2026-10-18T17:44:15.024358+00:00
1,Siemens Limited,Audited,Non-cumulative,Non-Consolidated,Ind-AS New,Quarterly,30-Sep-2022,Fourth Quarter,https://nsearchives.nseindia.com/corporate/xbr...,25-Nov-2022 12:32:04,...,2022Q3,ISIN:INE003A01024,EXACT_COMPANY_NAME_REVIEW,Siemens Ltd.,Capital Goods,SIEMENS,EQ,INE003A01024,https://nsearchives.nseindia.com/content/indic...,2026-10-18T17:44:15.024358+00:00
2,GMR AIRPORTS LIMITED,Un-Audited,Non-cumulative,Non-Consolidated,Ind-AS New,Quarterly,30-Sep-2022,Second Quarter,https://nsearchives.nseindia.com/corporate/xbr...,18-Nov-2022 11:09:29,...,2022Q3,ISIN:INE776C01039,EXACT_COMPANY_NAME_REVIEW,GMR Airports Ltd.,Services,GMRAIRPORT,EQ,INE776C01039,https://nsearchives.nseindia.com/content/indic...,2026-10-18T17:44:15.024358+00:00
3,Syrma SGS Technology Limited,Un-Audited,Non-cumulative,Non-Consolidated,Ind-AS New,Quarterly,30-Sep-2022,Second Quarter,https://nsearchives.nseindia.com/corporate/xbr...,16-Nov-2022 14:59:15,...,2022Q3,ISIN:INE0DYJ01015,EXACT_COMPANY_NAME_REVIEW,Syrma SGS Technology Ltd.,Capital Goods,SYRMA,EQ,INE0DYJ01015,https://nsearchives.nseindia.com/content/indic...,2026-10-18T17:44:15.024358+00:00
4,Tata Elxsi Limited,Audited,Non-cumulative,Non-Consolidated,Ind-AS New,Quarterly,30-Sep-2022,Second Quarter,https://nsearchives.nseindia.com/corporate/xbr...,16-Nov-2022 12:45:39,...,2022Q3,ISIN:INE670A01012,EXACT_COMPANY_NAME_REVIEW,Tata Elxsi Ltd.,Information Technology,TATAELXSI,EQ,INE670A01012,https://nsearchives.nseindia.com/content/indic...,2026-10-18T17:44:15.024358+00:00


## 4. Download source financial XBRL files

Potentially thousands of files: this can take a substantial amount of time.
Start with MAX_DOWNLOADS=5 to check access, then set it to None and rerun.
Existing valid files are reused; the log is saved after each attempt.
HTTP 401/403/429 stops the batch. Downloaded files remain available for extraction.
No access workaround is attempted. Use downloads according to applicable terms.
For persistence across Colab runtime resets, set ROOT to your mounted Drive.


In [ ]:
download_log = download_filings(
    selected_filings,
    ROOT / "xml",
    ROOT / "download_log.csv",
    max_downloads=None,
    pause_seconds=2,
    timeout_seconds=15,
    max_consecutive_failures=3,
)

display(download_log["DOWNLOAD_STATUS"].value_counts().to_frame("files"))

Unique XML URLs to process: 4,744

[1/4744] Starting: https://nsearchives.nseindia.com/corporate/xbrl/INDAS_911979_5911_03122022152749_WEB.xml
Result: CACHED_VALID_XML
Processed: 1/4744 | Successful: 1 | Failed: 0

[2/4744] Starting: https://nsearchives.nseindia.com/corporate/xbrl/INDAS_899560_4797_25112022123204_WEB.xml
Result: CACHED_VALID_XML
Processed: 2/4744 | Successful: 2 | Failed: 0

[3/4744] Starting: https://nsearchives.nseindia.com/corporate/xbrl/INDAS_885383_6841_18112022110929_WEB.xml
Result: CACHED_VALID_XML
Processed: 3/4744 | Successful: 3 | Failed: 0

[4/4744] Starting: https://nsearchives.nseindia.com/corporate/xbrl/INDAS_881082_6646_16112022145915_WEB.xml
Result: CACHED_VALID_XML
Processed: 4/4744 | Successful: 4 | Failed: 0

[5/4744] Starting: https://nsearchives.nseindia.com/corporate/xbrl/INDAS_880564_4737_16112022124539_WEB.xml
Result: CACHED_VALID_XML
Processed: 5/4744 | Successful: 5 | Failed: 0

[6/4744] Starting: https://nsearchives.nseindia.com/corporate/xbr

## 5. Load the conservative financial XBRL parser

The parser is copied from your existing nse-financial-research project.
It uses explicit concept mappings, unit definitions and XBRL contexts. It does
not multiply values again by the rounding label. Missing or conflicting values
remain missing. Undefined contexts are retained for inspection and excluded
from the strict panel. Banking fields are handled separately; bank income is
not substituted for industrial revenue. Unsupported concepts stay unfilled.


In [ ]:
"""Conservative NSE XBRL instance extraction; embedded into the Colab notebook."""
import io
import json
import re
import xml.etree.ElementTree as ET
from datetime import date
from decimal import Decimal, InvalidOperation
from pathlib import Path

XBRLI = 'http://www.xbrl.org/2003/instance'
XSI = 'http://www.w3.org/2001/XMLSchema-instance'
XLINK = 'http://www.w3.org/1999/xlink'
KNOWN_FORMATS = {'INDAS', 'NBFC', 'BANKING', 'NONINDAS'}
BASE_MAP = {
    'PAT': ['ProfitLossForPeriod', 'ProfitLossForThePeriod'],
    'PAT_CONTINUING_OPERATIONS': ['ProfitLossForPeriodFromContinuingOperations', 'ProfitLossForThePeriodFromContinuingOperations'],
    'PAT_DISCONTINUED_OPERATIONS': ['ProfitLossFromDiscontinuedOperationsAfterTax', 'ProfitLossFromDiscontinuingOperationAfterTax'],
    'PAT_OWNERS_OF_PARENT': ['ProfitOrLossAttributableToOwnersOfParent'],
    'PAT_NONCONTROLLING_INTERESTS': ['ProfitOrLossAttributableToNonControllingInterests'],
    'REVENUE': ['RevenueFromOperations'],
    'TOTAL_INCOME': ['Income'],
    'OTHER_INCOME': ['OtherIncome'],
    'PROFIT_BEFORE_TAX': ['ProfitBeforeTax'],
    'TAX_EXPENSE': ['TaxExpense'],
    'EXPENSES': ['Expenses'],
    'FINANCE_COSTS': ['FinanceCosts'],
    'EPS_BASIC': ['BasicEarningsLossPerShareFromContinuingAndDiscontinuedOperations'],
    'EPS_DILUTED': ['DilutedEarningsLossPerShareFromContinuingAndDiscontinuedOperations'],
    'ASSETS': ['Assets'], 'LIABILITIES': ['Liabilities'], 'EQUITY': ['Equity'],
    'BORROWINGS': ['Borrowings'],
}
FORMAT_MAPS = {kind: {k: list(v) for k, v in BASE_MAP.items()} for kind in KNOWN_FORMATS}
FORMAT_MAPS['NONINDAS']['TOTAL_INCOME'] = ['Revenue']
FORMAT_MAPS['BANKING'].update({
    'PROFIT_BEFORE_TAX': ['ProfitLossFromOrdinaryActivitiesBeforeTax'],
    'OPERATING_PROFIT_BEFORE_PROVISIONS': ['OperatingProfitBeforeProvisionAndContingencies'],
    'INTEREST_EARNED': ['InterestEarned'], 'INTEREST_EXPENDED': ['InterestExpended'],
    'PROVISIONS': ['ProvisionsOtherThanTaxAndContingencies'],
    'EPS_BASIC': ['BasicEarningsPerShareAfterExtraordinaryItems'],
    'EPS_DILUTED': ['DilutedEarningsPerShareAfterExtraordinaryItems'],
    'GROSS_NPA': ['GrossNonPerformingAssets'], 'NET_NPA': ['NonPerformingAssets'],
})
METADATA_MAP = {
    'NSE_SYMBOL': ['Symbol'], 'COMPANY_NAME': ['NameOfCompany', 'NameOfBank'],
    'STANDALONE_CONSOLIDATED': ['NatureOfReportStandaloneConsolidated'],
    'AUDITED_FLAG': ['WhetherResultsAreAuditedOrUnaudited'],
    'REPORTED_QUARTER_LABEL': ['ReportingQuarter'],
    'REPORTING_CURRENCY': ['DescriptionOfPresentationCurrency'],
    'ROUNDING': ['LevelOfRoundingUsedInFinancialStatements'],
    'REPORTED_START': ['DateOfStartOfReportingPeriod'],
    'REPORTED_END': ['DateOfEndOfReportingPeriod'],
    'FINANCIAL_YEAR_START': ['DateOfStartOfFinancialYear'],
    'FINANCIAL_YEAR_END': ['DateOfEndOfFinancialYear'],
}


def local_name(tag):
    return tag.rsplit('}', 1)[-1]


def namespace(tag):
    return tag[1:].split('}', 1)[0] if tag.startswith('{') else ''


def as_number(text):
    try:
        value = Decimal(text.strip())
        return value if value.is_finite() else None
    except (InvalidOperation, AttributeError):
        return None


def safe_date(text):
    try:
        return date.fromisoformat(text[:10])
    except (TypeError, ValueError):
        return None


def detect_format(filename, root):
    prefix = re.split(r'[_\-.]', Path(filename).name.upper())[0]
    named = prefix if prefix in KNOWN_FORMATS else None
    concepts = {local_name(e.tag) for e in root.iter() if e.get('contextRef')}
    if 'OperatingProfitBeforeProvisionAndContingencies' in concepts:
        content = 'BANKING'
    elif 'ProfitLossForPeriodBeforeMinorityInterest' in concepts:
        content = 'NONINDAS'
    elif {'Loans', 'ImpairmentOnFinancialInstruments'} <= concepts:
        content = 'NBFC'
    elif 'ProfitLossForPeriod' in concepts:
        content = 'INDAS_OR_NBFC'
    else:
        content = 'UNKNOWN'
    conflict = bool(named and content in KNOWN_FORMATS and named != content)
    detected = 'UNKNOWN' if conflict else named or (content if content in KNOWN_FORMATS else 'UNKNOWN')
    return detected, f'filename={named or "unknown"}; content={content}', conflict


def parse_xbrl(filename, payload):
    # Preserve evidence rather than resolving ambiguous facts by document order.
    if re.search(br'<!\s*(?:DOCTYPE|ENTITY)\b', payload, flags=re.I):
        raise ValueError('DTD/entity declarations are unsupported.')
    ns = dict(pair for _, pair in ET.iterparse(io.BytesIO(payload), events=['start-ns']))
    root = ET.fromstring(payload)
    if root.tag != f'{{{XBRLI}}}xbrl':
        raise ValueError('Expected an XBRL instance XML. Inline XBRL/HTML is not supported.')
    def expand_qname(text):
        prefix, sep, name = (text or '').partition(':')
        return f'{{{ns.get(prefix, prefix)}}}{name}' if sep else f'{{{ns.get("", "")}}}{prefix}'
    fmt, evidence, conflict = detect_format(filename, root)
    issues = []
    def issue(kind, context='', field='', detail=''):
        issues.append({'SOURCE_FILE': filename, 'CONTEXT_ID': context,
                       'FIELD': field, 'ISSUE': kind, 'DETAIL': detail})
    if fmt == 'UNKNOWN':
        issue('UNKNOWN_OR_CONFLICTING_FORMAT', detail=evidence)
    contexts = {}
    for element in root.findall(f'{{{XBRLI}}}context'):
        cid = element.get('id')
        if cid in contexts:
            raise ValueError(f'Duplicate context ID: {cid}')
        def text_at(path):
            found = element.find(path)
            return found.text.strip() if found is not None and found.text else None
        dimensions = []
        for member in element.iter():
            if local_name(member.tag) in {'explicitMember', 'typedMember'}:
                dimensions.append({'dimension': member.get('dimension'),
                                   'member': ''.join(member.itertext()).strip()})
        containers = element.findall(f'.//{{{XBRLI}}}segment') + element.findall(f'.//{{{XBRLI}}}scenario')
        contexts[cid] = {
            'ENTITY_IDENTIFIER': text_at(f'{{{XBRLI}}}entity/{{{XBRLI}}}identifier'),
            'PERIOD_START': text_at(f'{{{XBRLI}}}period/{{{XBRLI}}}startDate'),
            'PERIOD_END': text_at(f'{{{XBRLI}}}period/{{{XBRLI}}}endDate'),
            'INSTANT': text_at(f'{{{XBRLI}}}period/{{{XBRLI}}}instant'),
            'DIMENSIONS': json.dumps(dimensions, sort_keys=True),
            'HAS_CONTEXT_DETAIL': any(len(c) > 0 for c in containers),
        }
    units = {}
    for element in root.findall(f'{{{XBRLI}}}unit'):
        uid = element.get('id')
        if uid in units:
            raise ValueError(f'Duplicate unit ID: {uid}')
        divide = element.find(f'{{{XBRLI}}}divide')
        if divide is None:
            numerator = [expand_qname(e.text) for e in element.findall(f'{{{XBRLI}}}measure')]
            denominator = []
        else:
            numerator = [expand_qname(e.text) for e in divide.findall(f'{{{XBRLI}}}unitNumerator/{{{XBRLI}}}measure')]
            denominator = [expand_qname(e.text) for e in divide.findall(f'{{{XBRLI}}}unitDenominator/{{{XBRLI}}}measure')]
        units[uid] = {'numerator': sorted(numerator), 'denominator': sorted(denominator)}
    facts = []
    for index, element in enumerate(root.iter()):
        cid = element.get('contextRef')
        if not cid:
            continue
        text = ''.join(element.itertext()).strip()
        nil = element.get(f'{{{XSI}}}nil', '').lower() in {'true', '1'}
        unit = element.get('unitRef')
        facts.append({'SOURCE_FILE': filename, 'FORMAT': fmt, 'FACT_INDEX': index,
                      'CONCEPT': local_name(element.tag), 'CONCEPT_QNAME': element.tag,
                      'NAMESPACE': namespace(element.tag), 'CONTEXT_ID': cid,
                      'RAW_VALUE': text, 'IS_NIL': nil, 'UNIT_REF': unit,
                      'UNIT_DEFINITION': json.dumps(units.get(unit), sort_keys=True),
                      'DECIMALS': element.get('decimals'),
                      'NUMERIC_VALUE': str(as_number(text)) if not nil and as_number(text) is not None else None,
                      'CONTEXT_DEFINED': cid in contexts})
    groups = {}
    for fact in facts:
        groups.setdefault(fact['CONTEXT_ID'], []).append(fact)
    rows = []
    known_namespace = lambda f: bool(re.search(r'https?://(?:www\.)?(?:bseindia|nseindia)\.com/', f['NAMESPACE']))
    def pick(group, aliases, cid, field, numeric=False):
        # Alias ordering expresses semantic preference, not a fuzzy tag match.
        for alias in aliases:
            matches = [f for f in group if f['CONCEPT'] == alias and not f['IS_NIL'] and known_namespace(f)]
            if not matches:
                continue
            comparable = {(f['NUMERIC_VALUE'] if numeric else f['RAW_VALUE'], f['CONCEPT_QNAME'],
                           f['UNIT_DEFINITION'] if numeric else '') for f in matches}
            if len(comparable) != 1:
                issue('CONFLICTING_FACTS', cid, field, alias)
                return None
            fact = matches[0]
            if numeric and fact['NUMERIC_VALUE'] is None:
                issue('NONNUMERIC_MAPPED_FACT', cid, field, fact['RAW_VALUE'])
                return None
            if len(matches) > 1:
                issue('IDENTICAL_DUPLICATE_FACTS', cid, field, alias)
            return fact
        return None
    for cid, group in groups.items():
        if not any(f['NUMERIC_VALUE'] is not None and f['UNIT_REF'] for f in group):
            continue
        ctx = contexts.get(cid)
        row = {'SOURCE_FILE': filename, 'FILING_FORMAT': fmt, 'FORMAT_EVIDENCE': evidence,
               'CONTEXT_ID': cid, 'CONTEXT_DEFINED': ctx is not None}
        row.update(ctx or {'ENTITY_IDENTIFIER': None, 'PERIOD_START': None, 'PERIOD_END': None,
                           'INSTANT': None, 'DIMENSIONS': 'UNKNOWN', 'HAS_CONTEXT_DETAIL': True})
        for field, aliases in METADATA_MAP.items():
            chosen = pick(group, aliases, cid, field)
            row[field] = chosen['RAW_VALUE'] if chosen else None
        # Only document-wide identity/currency/rounding are allowed to fall back.
        # Audit status and reporting basis are never borrowed from another period.
        for field in ['NSE_SYMBOL', 'COMPANY_NAME', 'REPORTING_CURRENCY', 'ROUNDING']:
            if row[field] is None:
                chosen = pick(facts, METADATA_MAP[field], cid, field)
                row[field] = chosen['RAW_VALUE'] if chosen else None
        for target, source in [('PERIOD_START', 'REPORTED_START'), ('PERIOD_END', 'REPORTED_END')]:
            if not row[target] and row[source]:
                row[target] = row[source]
                row[target + '_SOURCE'] = 'reporting-date fact; context missing or incomplete'
            elif row[target]:
                row[target + '_SOURCE'] = 'context'
            if row[target] and row[source] and row[target] != row[source]:
                issue('DATE_CONFLICT', cid, target, f"context={row[target]}; fact={row[source]}")
        if row['NSE_SYMBOL'] is None and ctx and ctx['ENTITY_IDENTIFIER']:
            # Identifier is retained separately; do not assume it is an NSE symbol.
            issue('SYMBOL_MISSING', cid, detail=ctx['ENTITY_IDENTIFIER'])
        if ctx is None:
            issue('UNDEFINED_CONTEXT', cid, detail='Values retained for inspection; excluded from strict quarterly output.')
        field_facts = {}
        for field, aliases in FORMAT_MAPS.get(fmt, {}).items():
            chosen = pick(group, aliases, cid, field, numeric=True)
            row[field] = None
            row[field + '_TAG'] = chosen['CONCEPT_QNAME'] if chosen else None
            row[field + '_UNIT'] = chosen['UNIT_DEFINITION'] if chosen else None
            row[field + '_DECIMALS'] = chosen['DECIMALS'] if chosen else None
            if chosen:
                expected_eps = field.startswith('EPS_')
                definition = units.get(chosen['UNIT_REF'])
                num = definition['numerator'] if definition else []
                den = definition['denominator'] if definition else []
                currency = (len(num) == 1 and re.fullmatch(r'\{http://www.xbrl.org/2003/iso4217\}[A-Z]{3}', num[0]))
                correct_unit = currency and ((den == [f'{{{XBRLI}}}shares']) if expected_eps else not den)
                if not correct_unit:
                    issue('UNSUPPORTED_OR_MISSING_UNIT', cid, field, chosen['UNIT_DEFINITION'])
                else:
                    row[field] = chosen['NUMERIC_VALUE']
                    field_facts[field] = chosen
        start, end = safe_date(row['PERIOD_START']), safe_date(row['PERIOD_END'])
        row['DURATION_DAYS'] = (end - start).days + 1 if start and end else None
        row['PERIOD_KIND'] = ('instant' if row['INSTANT'] else 'quarter_like' if row['DURATION_DAYS'] is not None
                              and 70 <= row['DURATION_DAYS'] <= 110 else 'duration_other_or_unknown')
        row['PAT_DEFINITION'] = 'Total profit/loss for period; not owners-only or comprehensive income'
        if row.get('PAT') is None and row.get('PROFIT_BEFORE_TAX') is not None:
            issue('TOTAL_PAT_UNAVAILABLE', cid, 'PAT', 'Review raw facts; no substitution from continuing operations or PBT minus tax.')
        if all(row.get(k) is not None for k in ['PROFIT_BEFORE_TAX', 'TAX_EXPENSE', 'PAT']):
            definitions = {row[k + '_UNIT'] for k in ['PROFIT_BEFORE_TAX', 'TAX_EXPENSE', 'PAT']}
            if len(definitions) == 1:
                difference = Decimal(row['PROFIT_BEFORE_TAX']) - Decimal(row['TAX_EXPENSE']) - Decimal(row['PAT'])
                row['PBT_MINUS_TAX_MINUS_PAT'] = str(difference)
                if difference:
                    issue('PAT_RECONCILIATION_DIFFERENCE', cid, 'PAT', str(difference))
        row['STRICT_QUARTER_CANDIDATE'] = bool(ctx is not None and not row['HAS_CONTEXT_DETAIL']
            and row['PERIOD_KIND'] == 'quarter_like' and fmt in KNOWN_FORMATS
            and (row['STANDALONE_CONSOLIDATED'] or '').strip().lower() in {'standalone', 'consolidated'}
            and row.get('PAT') is not None and row.get('NSE_SYMBOL'))
        if row['STRICT_QUARTER_CANDIDATE'] and any(i['CONTEXT_ID'] == cid and i['ISSUE'] in
            {'DATE_CONFLICT', 'CONFLICTING_FACTS'} for i in issues):
            row['STRICT_QUARTER_CANDIDATE'] = False
        rows.append(row)
    # Financial statement concepts that need mapping remain visible, never guessed.
    mapped = {a for aliases in FORMAT_MAPS.get(fmt, {}).values() for a in aliases}
    unmapped = sorted({f['CONCEPT'] for f in facts if f['NUMERIC_VALUE'] is not None and
                       any(t in f['CONCEPT'].lower() for t in ['profit', 'tax', 'revenue', 'earning'])
                       and f['CONCEPT'] not in mapped})
    summary = {'SOURCE_FILE': filename, 'FORMAT': fmt, 'FORMAT_EVIDENCE': evidence,
               'FACTS': len(facts), 'DEFINED_CONTEXTS': len(contexts), 'EXTRACTED_CONTEXT_ROWS': len(rows),
               'UNDEFINED_CONTEXT_REFS': len({f['CONTEXT_ID'] for f in facts if not f['CONTEXT_DEFINED']}),
               'PAT_CONTEXT_ROWS': sum(r.get('PAT') is not None for r in rows),
               'STRICT_QUARTER_CANDIDATES': sum(r['STRICT_QUARTER_CANDIDATE'] for r in rows),
               'UNMAPPED_RELATED_CONCEPTS': '|'.join(unmapped)}
    return rows, facts, issues, summary


## 6. Extract financial values and retain provenance

Produces all extracted context rows, filing summaries, extraction errors and
issues. This stage reads only successfully downloaded/cached XMLs.


In [ ]:
all_contexts = extract_downloads(
    download_log, selected_filings, parse_xbrl, ROOT / "extracted")
print("Extracted financial context rows:", len(all_contexts))
display(all_contexts.head())


## 7. Build the standalone quarterly dataset and coverage report

Strict rows must have a defined nondimensional context, usable PAT, correct
period length and standalone basis. Their dates must match filing metadata.
Exact ISIN matches enter the identity screen directly. Name/symbol matches
must also match the symbol reported inside the XML. Multiple surviving filings/contexts for a company-quarter are all
excluded pending review; the notebook does not silently choose a revision.
Coverage includes every company in the index snapshot, even those with no data.
The strict candidate flag is a screening rule, not a complete accounting audit.


In [ ]:
quarterly_data, complete_data, company_summary, duplicate_review = build_panel(
    all_contexts, nifty500, ANALYSIS_START, ANALYSIS_END, REQUIRED_FIELDS)

quarterly_data.to_csv(ROOT / "Nifty500_Standalone_Quarterly_Available.csv", index=False)
complete_data.to_csv(ROOT / "Nifty500_Complete_Quarterly_Panel.csv", index=False)
company_summary.to_csv(ROOT / "Nifty500_Company_Coverage.csv")
duplicate_review.to_csv(ROOT / "duplicate_company_quarters_review.csv", index=False)

print("Required calendar quarters:", len(pd.period_range(ANALYSIS_START, ANALYSIS_END, freq="Q")))
print("Required financial fields:", REQUIRED_FIELDS)
print("Companies with all required quarters:", int(company_summary.eligible.sum()))
print("Complete panel records:", len(complete_data))
display(company_summary)


## 8. Preliminary descriptive analysis

Available-row statistics can describe an unbalanced sample. Use complete_data
for comparisons requiring a balanced panel. Report the selected sample size.
PAT can be zero or negative; neither is removed automatically. Do not combine
currencies/units or banking and industrial definitions without checking them.
No hypothesis tests or predictive modelling are claimed by this notebook.


In [ ]:
if quarterly_data.empty:
    print("No strict quarterly rows. Inspect downloads, contexts, identity and duplicate reports.")
else:
    available_complete = quarterly_data.loc[quarterly_data[REQUIRED_FIELDS].notna().all(axis=1)]
    display(available_complete.groupby("_quarter").agg(
        records=("COMPANY_KEY", "size"), companies=("COMPANY_KEY", "nunique")))
    display(available_complete[REQUIRED_FIELDS].describe().T)
    print("These describe available observations; they do not prove a balanced 200-company panel.")


## 9. Download all outputs, XMLs and logs

The ZIP contains the source snapshot and source filings used in your analysis.
Inspect applicable data usage/redistribution terms before publishing it.


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(str(ROOT.parent / "Nifty500_Financial_Results"), "zip", ROOT)
files.download(archive)
